In [1]:
"""
1. Post-Training Quantization (PTQ) — ResNet-50 / CIFAR-10
===========================================================
Takes a trained FP32 model and applies quantization AFTER training.
No gradient updates — only calibration of scale S and zero-point Z.

Mathematical foundation:
  Q(x)  = round(x / S) + Z          [quantization — maps float → integer]
  x̂     = S · (Q(x) − Z)            [dequantization — reconstruction at inference]
  ε     = x − x̂,  |ε| ≤ S/2        [quantization error — bounded by step size]
  S     = (x_max − x_min) / (2^b−1) [scale — divides observed range over all levels]
  Z     = round(−x_min / S)         [zero-point — maps real 0 to an integer]

  INT8: 2^8 = 256 levels, S is small → fine-grained, error small
  INT4: 2^4 = 16 levels,  S is large → coarse, error accumulates per layer

Three back-ends benchmarked:
  1. Dynamic      — Linear weights → qint8; activations FP32 (no calibration)
  2. Static (FX)  — Conv2d + Linear weights + activations → INT8
                    Sweeps 4 observer types: minmax, moving_avg, histogram, percentile
  3. FX-Graph     — torch.ao graph-mode with x86/fbgemm default qconfigs; broadest op coverage

Output: __1__PTQ.json
"""

'\n1. Post-Training Quantization (PTQ) — ResNet-50 / CIFAR-10\n===========================================================\nTakes a trained FP32 model and applies quantization AFTER training.\nNo gradient updates — only calibration of scale S and zero-point Z.\n\nMathematical foundation:\n  Q(x)  = round(x / S) + Z          [quantization — maps float → integer]\n  x̂     = S · (Q(x) − Z)            [dequantization — reconstruction at inference]\n  ε     = x − x̂,  |ε| ≤ S/2        [quantization error — bounded by step size]\n  S     = (x_max − x_min) / (2^b−1) [scale — divides observed range over all levels]\n  Z     = round(−x_min / S)         [zero-point — maps real 0 to an integer]\n\n  INT8: 2^8 = 256 levels, S is small → fine-grained, error small\n  INT4: 2^4 = 16 levels,  S is large → coarse, error accumulates per layer\n\nThree back-ends benchmarked:\n  1. Dynamic      — Linear weights → qint8; activations FP32 (no calibration)\n  2. Static (FX)  — Conv2d + Linear weights + acti

In [2]:
import os, json, time, copy, tempfile, warnings
import numpy as np
import torch
import torch.nn as nn
import torch.quantization as tq
from torch.ao.quantization import get_default_qconfig
from torch.ao.quantization.quantize_fx import prepare_fx, convert_fx
import torchvision
import torchvision.transforms as transforms
import torchvision.models as models
from torch.utils.data import DataLoader, Subset
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score

warnings.filterwarnings("ignore")

# ── Config ────────────────────────────────────────────────────────────────────
BASELINE_MODEL_PATH   = "../__2__baseline_resnet50_cifar10.pth"
BASELINE_METRICS_PATH = "../__2__baseline_metrics.json"
OUTPUT_JSON           = "__1__PTQ.json"

DEVICE         = torch.device("cpu")   # PyTorch static PTQ is CPU-only
BATCH_SIZE     = 128
NUM_WORKERS    = 2
NUM_CLASSES    = 10
CALIB_SIZE     = 1024    # images used to calibrate static observers
CALIB_BATCHES  = 8       # max forward passes during calibration
INFERENCE_RUNS = 50      # repetitions for latency measurement

CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD  = (0.2023, 0.1994, 0.2010)

In [3]:
# ── Observer configs for static PTQ sweep ─────────────────────────────────────
# Different observers use different strategies to estimate x_min / x_max,
# which directly determines S = (x_max - x_min) / (2^b - 1)
from torch.ao.quantization import QConfig

OBSERVER_QCONFIGS = {
    "minmax": QConfig(
        activation=tq.MinMaxObserver.with_args(
            dtype=torch.quint8, qscheme=torch.per_tensor_affine),
        weight=tq.PerChannelMinMaxObserver.with_args(
            dtype=torch.qint8, qscheme=torch.per_channel_symmetric),
    ),
    "moving_avg": QConfig(
        activation=tq.MovingAverageMinMaxObserver.with_args(
            dtype=torch.quint8, qscheme=torch.per_tensor_affine),
        weight=tq.MovingAveragePerChannelMinMaxObserver.with_args(
            dtype=torch.qint8, qscheme=torch.per_channel_symmetric),
    ),
    "histogram": QConfig(
        activation=tq.HistogramObserver.with_args(
            dtype=torch.quint8, qscheme=torch.per_tensor_affine),
        weight=tq.PerChannelMinMaxObserver.with_args(
            dtype=torch.qint8, qscheme=torch.per_channel_symmetric),
    ),
    "percentile": QConfig(
        activation=tq.HistogramObserver.with_args(
            dtype=torch.quint8, qscheme=torch.per_tensor_affine,
            reduce_range=True),
        weight=tq.PerChannelMinMaxObserver.with_args(
            dtype=torch.qint8, qscheme=torch.per_channel_symmetric),
    ),
}

In [4]:
# ── Model ─────────────────────────────────────────────────────────────────────
def build_model(num_classes: int = 10) -> nn.Module:
    model = models.resnet50(weights=None)
    model.conv1   = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    model.maxpool = nn.Identity()
    model.fc      = nn.Linear(model.fc.in_features, num_classes)
    return model

def load_baseline(path: str) -> nn.Module:
    model = build_model(NUM_CLASSES).cpu()
    model.load_state_dict(torch.load(path, map_location="cpu"))
    model.eval()
    return model


# ── Data ──────────────────────────────────────────────────────────────────────
def get_test_loader() -> DataLoader:
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
    ])
    ds = torchvision.datasets.CIFAR10(root="../data", train=False,
                                       download=True, transform=transform)
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False,
                      num_workers=NUM_WORKERS, pin_memory=False)

def get_calib_loader() -> DataLoader:
    """
    Small representative subset from training split.
    Observers watch this data to compute x_min and x_max,
    from which S = (x_max - x_min) / 255 is derived.
    """
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
    ])
    ds  = torchvision.datasets.CIFAR10(root="../data", train=True,
                                        download=True, transform=transform)
    sub = Subset(ds, list(range(CALIB_SIZE)))
    return DataLoader(sub, batch_size=BATCH_SIZE, shuffle=False,
                      num_workers=NUM_WORKERS, pin_memory=False)

In [5]:
# ── Utilities ─────────────────────────────────────────────────────────────────
def disk_size_mb(model: nn.Module) -> float:
    with tempfile.NamedTemporaryFile(suffix=".pth", delete=False) as f:
        tmp = f.name
    try:
        # Use torch.save on the full model (not just state_dict) for quantized
        # models — state_dict() alone may not preserve INT8 tensor packing.
        torch.save(model, tmp)
        return os.path.getsize(tmp) / 1024 ** 2
    finally:
        os.remove(tmp)

def fp32_ram_mb(model: nn.Module) -> float:
    return sum(p.numel() for p in model.parameters()) * 4 / 1024 ** 2

@torch.no_grad()
def evaluate(model: nn.Module, loader: DataLoader) -> dict:
    model.eval()
    preds, labels = [], []
    for inputs, lbls in loader:
        preds.extend(model(inputs).argmax(1).numpy())
        labels.extend(lbls.numpy())
    y_pred, y_true = np.array(preds), np.array(labels)
    return {
        "accuracy" : float(accuracy_score(y_true, y_pred)),
        "precision": float(precision_score(y_true, y_pred, average="macro", zero_division=0)),
        "recall"   : float(recall_score(y_true, y_pred, average="macro", zero_division=0)),
        "f1"       : float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
    }

def measure_cpu_ms(model: nn.Module, n: int = INFERENCE_RUNS) -> float:
    model = model.cpu().eval()
    dummy = torch.randn(BATCH_SIZE, 3, 32, 32)
    with torch.no_grad():
        for _ in range(5):
            model(dummy)
        t0 = time.perf_counter()
        for _ in range(n):
            model(dummy)
    return (time.perf_counter() - t0) / n * 1000

def build_row(backend, observer, description, metrics, baseline_acc,
              base_disk, disk_mb, cpu_ms, layers_quantized, **extra) -> dict:
    return {
        "backend"           : backend,
        "observer"          : observer,
        "description"       : description,
        "accuracy"          : round(metrics["accuracy"],  6),
        "precision"         : round(metrics["precision"], 6),
        "recall"            : round(metrics["recall"],    6),
        "f1"                : round(metrics["f1"],        6),
        "accuracy_drop"     : round(baseline_acc - metrics["accuracy"], 6),
        "size_disk_mb"      : round(disk_mb, 4),
        "disk_saved_mb"     : round(base_disk - disk_mb, 4),
        "compression_ratio" : round(base_disk / disk_mb, 4) if disk_mb else None,
        "inference_cpu_ms"  : round(cpu_ms, 4),
        "layers_quantized"  : layers_quantized,
        **extra,
    }

@torch.no_grad()
def calibrate(model: nn.Module, loader: DataLoader,
              max_batches: int = CALIB_BATCHES) -> None:
    """
    Calibration pass: runs representative data so observers can collect
    activation statistics (x_min, x_max) and compute S and Z.
    S = (x_max - x_min) / (2^b - 1)
    Z = round(-x_min / S)
    """
    model.eval()
    for i, (inputs, _) in enumerate(loader):
        model(inputs)
        if i + 1 >= max_batches:
            break


In [6]:
# ══════════════════════════════════════════════════════════════════════════════
# 1. Dynamic PTQ
# ──────────────────────────────────────────────────────────────────────────────
# What:    Weights of nn.Linear layers → qint8
# S/Z:     Computed on-the-fly per inference batch (not from calibration data)
# Formula: Q(x) = round(x/S) + Z  applied to Linear weight tensors only
# When:    No calibration data available; quick to apply; lower accuracy gain
#          than static because Conv2d stays FP32
# ══════════════════════════════════════════════════════════════════════════════
def run_dynamic_ptq(fp32_model, test_loader, baseline_acc, base_disk) -> dict:
    print("\n  [1/3] Dynamic PTQ")
    print("        Quantizes  : Linear weights → qint8")
    print("        Activations: FP32 (scale computed per batch, on-the-fly)")
    print("        Calibration: none required")

    q_model = torch.quantization.quantize_dynamic(
        copy.deepcopy(fp32_model).cpu(), {nn.Linear}, dtype=torch.qint8
    )
    q_model.eval()

    metrics = evaluate(q_model, test_loader)
    cpu_ms  = measure_cpu_ms(q_model)
    disk_mb = disk_size_mb(q_model)

    row = build_row(
        backend          = "dynamic",
        observer         = "per_channel_affine (runtime)",
        description      = (
            "Dynamic PTQ: only Linear weights quantized to qint8. "
            "S and Z computed on-the-fly per batch — no calibration data. "
            "Conv2d stays FP32. Fast to apply; no memory of x_min/x_max needed."
        ),
        metrics          = metrics,
        baseline_acc     = baseline_acc,
        base_disk        = base_disk,
        disk_mb          = disk_mb,
        cpu_ms           = cpu_ms,
        layers_quantized = "Linear only",
        calibration_samples = 0,
        quantization_formulas = {
            "Q(x)"       : "round(x / S) + Z",
            "x̂"          : "S · (Q(x) − Z)",
            "ε"          : "|x − x̂| ≤ S/2",
            "S"          : "computed per-batch at runtime from weight range",
            "dtype_w"    : "qint8  (8-bit signed integer)",
            "dtype_acts" : "float32 (not quantized)",
        },
    )
    print(f"        → Acc: {metrics['accuracy']:.4f}  "
          f"Drop: {baseline_acc - metrics['accuracy']:+.4f}  "
          f"Disk: {disk_mb:.2f} MB  CPU: {cpu_ms:.1f} ms")
    return row

In [7]:
# ══════════════════════════════════════════════════════════════════════════════
# 2. Static PTQ — FX-graph mode, sweeping 4 observer strategies
# ──────────────────────────────────────────────────────────────────────────────
# What:    Conv2d + Linear weights AND activations → INT8
# S/Z:     Determined ONCE from calibration data, then fixed for all inference
#          S = (x_max − x_min) / (2^b − 1)   [b=8 → 255 quantization levels]
#          Z = round(−x_min / S)
# Graph:   FX tracing fuses Conv-BN-ReLU and residual adds automatically
# Sweep:   4 observer types to compare x_min/x_max estimation strategies
# ══════════════════════════════════════════════════════════════════════════════
def run_static_ptq(fp32_model, test_loader, calib_loader,
                   baseline_acc, base_disk) -> list:
    print("\n  [2/3] Static PTQ — FX-graph mode  (sweeping 4 observers)")
    print("        Quantizes : Conv2d + Linear + residual adds (weights + activations → INT8)")
    print(f"        Scale S  = (x_max − x_min) / (2^8 − 1)  from {CALIB_SIZE} calib images")
    rows = []
    example = torch.randn(1, 3, 32, 32)

    for obs_name, qconfig in OBSERVER_QCONFIGS.items():
        print(f"        Observer: {obs_name:12s}", end="", flush=True)
        try:
            model    = copy.deepcopy(fp32_model).cpu().eval()
            prepared = prepare_fx(model, {"": qconfig}, example)
            calibrate(prepared, calib_loader)
            q_model  = convert_fx(prepared)
            q_model.eval()

            metrics = evaluate(q_model, test_loader)
            cpu_ms  = measure_cpu_ms(q_model)
            disk_mb = disk_size_mb(q_model)

            row = build_row(
                backend          = "static_fx",
                observer         = obs_name,
                description      = (
                    f"Static PTQ (FX-graph, {obs_name}): graph-traced fusion of "
                    f"Conv-BN-ReLU and residual adds. "
                    f"S and Z calibrated from {CALIB_SIZE} training images. "
                    "Weights and activations quantized to INT8."
                ),
                metrics          = metrics,
                baseline_acc     = baseline_acc,
                base_disk        = base_disk,
                disk_mb          = disk_mb,
                cpu_ms           = cpu_ms,
                layers_quantized = "Conv2d + Linear + residual adds (weights + activations)",
                calibration_samples = CALIB_SIZE,
                quantization_formulas = {
                    "S"        : "S = (x_max − x_min) / (2^8 − 1)  [255 levels for INT8]",
                    "Z"        : "Z = round(−x_min / S)",
                    "Q(x)"     : "Q(x) = round(x / S) + Z",
                    "x̂"        : "x̂ = S · (Q(x) − Z)",
                    "max_err"  : "|ε| ≤ S/2  →  smaller S = finer grid = less error",
                    "dtype_w"  : "qint8",
                    "dtype_act": "quint8",
                },
            )
            print(f" → Acc: {metrics['accuracy']:.4f}  "
                  f"Drop: {baseline_acc - metrics['accuracy']:+.4f}  "
                  f"Disk: {disk_mb:.2f} MB  CPU: {cpu_ms:.1f} ms")
        except Exception as e:
            print(f" → FAILED: {e}")
            row = {
                "backend"    : "static_fx",
                "observer"   : obs_name,
                "description": f"FAILED: {e}",
                "accuracy"   : None,
                "disk_saved_mb": None,
            }
        rows.append(row)
    return rows

In [8]:
# ══════════════════════════════════════════════════════════════════════════════
# 3. FX-Graph Static PTQ — torch.ao default qconfigs (x86 / fbgemm)
# ──────────────────────────────────────────────────────────────────────────────
# Same S/Z math as static sweep above, but uses torch.ao's recommended
# default qconfigs tuned for specific CPU backends:
#   x86    → targets AVX-512 VNNI (INT8 throughput ~4× FP32)
#   fbgemm → x86 fallback, also used by Facebook's inference stack
# ══════════════════════════════════════════════════════════════════════════════
def run_fx_ptq(fp32_model, test_loader, calib_loader,
               baseline_acc, base_disk) -> list:
    print("\n  [3/3] FX-Graph Static PTQ (torch.ao default configs) — x86 + fbgemm")
    print("        Uses backend-tuned qconfigs; broader op coverage than observer sweep")
    rows = []
    example = torch.randn(1, 3, 32, 32)

    for backend in ["x86", "fbgemm"]:
        print(f"        Backend: {backend:10s}", end="", flush=True)
        try:
            model    = copy.deepcopy(fp32_model).cpu().eval()
            prepared = prepare_fx(model, {"": get_default_qconfig(backend)}, example)
            calibrate(prepared, calib_loader)
            q_model  = convert_fx(prepared)
            q_model.eval()

            metrics = evaluate(q_model, test_loader)
            cpu_ms  = measure_cpu_ms(q_model)
            disk_mb = disk_size_mb(q_model)

            row = build_row(
                backend          = f"fx_static_{backend}",
                observer         = f"default_qconfig({backend})",
                description      = (
                    f"FX-graph static PTQ (backend={backend}): "
                    "torch.ao traces the computation graph, automatically fuses "
                    "Conv-BN-ReLU and quantizes all eligible ops including residual adds. "
                    "Same S/Z formulas as eager static; backend-tuned qconfig."
                ),
                metrics          = metrics,
                baseline_acc     = baseline_acc,
                base_disk        = base_disk,
                disk_mb          = disk_mb,
                cpu_ms           = cpu_ms,
                layers_quantized = "Conv2d + Linear + residual adds (auto-fused)",
                calibration_samples = CALIB_SIZE,
                quantization_formulas = {
                    "backend"    : backend,
                    "fusion"     : "automatic via FX graph tracing",
                    "dtype"      : "qint8 (weights), quint8 (activations)",
                    "throughput" : "INT8 VNNI ~4× FP32 on AVX-512 CPUs",
                },
            )
            print(f" → Acc: {metrics['accuracy']:.4f}  "
                  f"Drop: {baseline_acc - metrics['accuracy']:+.4f}  "
                  f"Disk: {disk_mb:.2f} MB  CPU: {cpu_ms:.1f} ms")
        except Exception as e:
            print(f" → FAILED: {e}")
            row = {
                "backend"      : f"fx_static_{backend}",
                "observer"     : f"default_qconfig({backend})",
                "description"  : f"FAILED: {str(e)}",
                "accuracy"     : None,
                "disk_saved_mb": None,
            }
        rows.append(row)
    return rows

In [9]:
# ── Main ──────────────────────────────────────────────────────────────────────
def main():
    print(f"\n{'='*60}")
    print("  2. Post-Training Quantization (PTQ) — ResNet-50 / CIFAR-10")
    print("  Math: Q(x)=round(x/S)+Z | S=(x_max-x_min)/(2^b-1) | |ε|≤S/2")
    print(f"  Device: CPU  |  Calibration: {CALIB_SIZE} images")
    print(f"{'='*60}")

    with open(BASELINE_METRICS_PATH) as f:
        baseline = json.load(f)
    baseline_acc = baseline["accuracy"]

    fp32_model   = load_baseline(BASELINE_MODEL_PATH)
    test_loader  = get_test_loader()
    calib_loader = get_calib_loader()

    base_disk = disk_size_mb(fp32_model)
    base_ram  = fp32_ram_mb(fp32_model)
    base_cpu  = measure_cpu_ms(fp32_model)
    print(f"\n  Baseline: Disk {base_disk:.2f} MB | RAM {base_ram:.2f} MB | CPU {base_cpu:.1f} ms")

    results = []
    results.append(run_dynamic_ptq(fp32_model, test_loader, baseline_acc, base_disk))
    results.extend(run_static_ptq(fp32_model, test_loader, calib_loader, baseline_acc, base_disk))
    results.extend(run_fx_ptq(fp32_model, test_loader, calib_loader, baseline_acc, base_disk))

    valid = [r for r in results if r.get("accuracy") is not None]
    if not valid:
        print("\n  ✗ All backends failed. Check PyTorch version compatibility.")
        return
    # Best = lowest accuracy drop; break ties by smallest disk size
    best = min(valid, key=lambda r: (r["accuracy_drop"], r["size_disk_mb"]))

    report = {
        "method"      : "post_training_quantization",
        "description" : (
            "PTQ: quantization applied after training with no gradient updates. "
            "Scale S and zero-point Z derived from calibration data. "
            "Three back-ends: dynamic (no calib), static FX sweep (4 observers), "
            "FX-graph with backend-tuned default qconfigs."
        ),
        "ptq_math": {
            "quantize"    : "Q(x) = round(x / S) + Z",
            "dequantize"  : "x̂ = S · (Q(x) − Z)",
            "error"       : "ε = x − x̂,  |ε| ≤ S/2",
            "scale"       : "S = (x_max − x_min) / (2^b − 1)",
            "zero_point"  : "Z = round(−x_min / S)",
            "int8_levels" : "2^8 − 1 = 255",
            "int4_levels" : "2^4 − 1 = 15",
            "note"        : (
                "INT8 works well with PTQ; INT4 degrades because 15 levels "
                "gives large S → large |ε|, and errors accumulate through layers "
                "with no mechanism to correct them (that is QAT's job)."
            ),
        },
        "calibration_samples" : CALIB_SIZE,
        "calibration_batches" : CALIB_BATCHES,
        "target_dtype"        : "qint8 (weights) / quint8 (activations)",
        "baseline"            : baseline,
        "baseline_size_breakdown": {
            "ram_fp32_mb"      : round(base_ram, 4),
            "disk_pth_mb"      : round(base_disk, 4),
            "inference_cpu_ms" : round(base_cpu, 4),
        },
        "device"     : "cpu",
        "best_config": {
            "backend"          : best["backend"],
            "observer"         : best["observer"],
            "accuracy"         : best["accuracy"],
            "accuracy_drop"    : best["accuracy_drop"],
            "size_disk_mb"     : best["size_disk_mb"],
            "compression_ratio": best.get("compression_ratio"),
        },
        "results": results,
        "notes": {
            "dynamic"      : "No calibration; Linear only; fastest to apply; Conv stays FP32",
            "static_fx"    : "Calibration required; best INT8 accuracy; Conv-BN-ReLU auto-fused by FX",
            "fx_default"   : "Backend-tuned qconfig; broadest op coverage; model must be torch.fx-traceable",
            "gpu_note"     : "PyTorch static PTQ is CPU-only; GPU INT8 → TensorRT / ONNX Runtime",
            "speedup_note" : "INT8 ops ~2-4× faster than FP32 on CPUs with AVX-512 VNNI support",
            "vs_qat"       : (
                "PTQ requires no retraining and is fast; QAT closes the accuracy gap at INT4 "
                "by inserting fake-quant nodes during training and using the STE trick."
            ),
            "disk_size_note": (
                "disk_size_mb uses torch.save(model) (full model, not state_dict) "
                "to ensure INT8 tensor packing is preserved in the serialized file."
            ),
        },
    }

    with open(OUTPUT_JSON, "w") as f:
        json.dump(report, f, indent=2)
    print(f"\n{'='*60}")
    print(f"  ✓ Saved → {OUTPUT_JSON}")
    print(f"  Best config : {best['backend']} | observer={best['observer']}")
    print(f"  Accuracy    : {best['accuracy']:.4f} (drop {best['accuracy_drop']:+.4f})")
    print(f"  Disk        : {best['size_disk_mb']:.2f} MB  "
          f"(saved {best.get('disk_saved_mb', 'N/A')} MB, "
          f"×{best.get('compression_ratio', '?')})")
    print(f"{'='*60}\n")

In [10]:
if __name__ == "__main__":
    main()


  2. Post-Training Quantization (PTQ) — ResNet-50 / CIFAR-10
  Math: Q(x)=round(x/S)+Z | S=(x_max-x_min)/(2^b-1) | |ε|≤S/2
  Device: CPU  |  Calibration: 1024 images

  Baseline: Disk 90.05 MB | RAM 89.72 MB | CPU 887.9 ms

  [1/3] Dynamic PTQ
        Quantizes  : Linear weights → qint8
        Activations: FP32 (scale computed per batch, on-the-fly)
        Calibration: none required
        → Acc: 0.9319  Drop: +0.0001  Disk: 89.99 MB  CPU: 1394.8 ms

  [2/3] Static PTQ — FX-graph mode  (sweeping 4 observers)
        Quantizes : Conv2d + Linear + residual adds (weights + activations → INT8)
        Scale S  = (x_max − x_min) / (2^8 − 1)  from 1024 calib images
        Observer: minmax      

W0330 12:20:21.784000 36552 Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


 → Acc: 0.9329  Drop: -0.0009  Disk: 22.99 MB  CPU: 501.5 ms
        Observer: moving_avg   → Acc: 0.9316  Drop: +0.0004  Disk: 22.99 MB  CPU: 457.7 ms
        Observer: histogram    → Acc: 0.9315  Drop: +0.0005  Disk: 22.99 MB  CPU: 226.9 ms
        Observer: percentile   → Acc: 0.9321  Drop: -0.0001  Disk: 22.99 MB  CPU: 371.8 ms

  [3/3] FX-Graph Static PTQ (torch.ao default configs) — x86 + fbgemm
        Uses backend-tuned qconfigs; broader op coverage than observer sweep
        Backend: x86        → Acc: 0.9321  Drop: -0.0001  Disk: 22.99 MB  CPU: 256.4 ms
        Backend: fbgemm     → Acc: 0.9321  Drop: -0.0001  Disk: 22.99 MB  CPU: 259.5 ms

  ✓ Saved → __1__PTQ.json
  Best config : static_fx | observer=minmax
  Accuracy    : 0.9329 (drop -0.0009)
  Disk        : 22.99 MB  (saved 67.0612 MB, ×3.917)



In [ ]:
"""
Save quantized models as .pt files for each PTQ method
"""
import os, copy
import torch
import torch.nn as nn

os.makedirs("__1__PTQ_quantized_models", exist_ok=True)


# ── Load baseline model first ─────────────────────────────────────────────────
fp32_model   = load_baseline(BASELINE_MODEL_PATH)
calib_loader = get_calib_loader()
example      = torch.randn(1, 3, 32, 32)
print(f"✓ Baseline loaded")


# ── 1. Dynamic PTQ ────────────────────────────────────────────────────────────
print("Saving dynamic PTQ model...")
q_dynamic = torch.quantization.quantize_dynamic(
    copy.deepcopy(fp32_model).cpu(), {nn.Linear}, dtype=torch.qint8
)
q_dynamic.eval()
torch.save(q_dynamic, "__1__PTQ_quantized_models/resnet50_dynamic.pt")
print("  ✓ __1__PTQ_quantized_models/resnet50_dynamic.pt")

# ── 2. Static PTQ — 4 observers ───────────────────────────────────────────────
print("\nSaving static FX PTQ models (4 observers)...")
example = torch.randn(1, 3, 32, 32)

for obs_name, qconfig in OBSERVER_QCONFIGS.items():
    try:
        model    = copy.deepcopy(fp32_model).cpu().eval()
        prepared = prepare_fx(model, {"": qconfig}, example)
        calibrate(prepared, calib_loader)
        q_model  = convert_fx(prepared)
        q_model.eval()
        torch.save(q_model, f"__1__PTQ_quantized_models/resnet50_static_{obs_name}.pt")
        print(f"  ✓ __1__PTQ_quantized_models/resnet50_static_{obs_name}.pt")
    except Exception as e:
        print(f"  ✗ {obs_name} FAILED: {e}")

# ── 3. FX-Graph — x86 + fbgemm ────────────────────────────────────────────────
print("\nSaving FX-graph PTQ models (x86 + fbgemm)...")
for backend in ["x86", "fbgemm"]:
    try:
        model    = copy.deepcopy(fp32_model).cpu().eval()
        prepared = prepare_fx(model, {"": get_default_qconfig(backend)}, example)
        calibrate(prepared, calib_loader)
        q_model  = convert_fx(prepared)
        q_model.eval()
        torch.save(q_model, f"__1__PTQ_quantized_models/resnet50_fx_{backend}.pt")
        print(f"  ✓ __1__PTQ_quantized_models/resnet50_fx_{backend}.pt")
    except Exception as e:
        print(f"  ✗ {backend} FAILED: {e}")

print("\nDone! Saved models:")
for f in sorted(os.listdir("__1__PTQ_quantized_models")):
    path = os.path.join("__1__PTQ_quantized_models", f)
    print(f"  {f:45s} {os.path.getsize(path)/1024**2:.2f} MB")

✓ Baseline loaded
Saving dynamic PTQ model...
  ✓ quantized_models/resnet50_dynamic.pt

Saving static FX PTQ models (4 observers)...
  ✓ quantized_models/resnet50_static_minmax.pt
  ✓ quantized_models/resnet50_static_moving_avg.pt
  ✓ quantized_models/resnet50_static_histogram.pt
  ✓ quantized_models/resnet50_static_percentile.pt

Saving FX-graph PTQ models (x86 + fbgemm)...
  ✓ quantized_models/resnet50_fx_x86.pt
  ✓ quantized_models/resnet50_fx_fbgemm.pt

Done! Saved models:
  resnet50_dynamic.pt                           89.99 MB
  resnet50_fx_fbgemm.pt                         22.99 MB
  resnet50_fx_x86.pt                            22.99 MB
  resnet50_static_histogram.pt                  22.99 MB
  resnet50_static_minmax.pt                     22.99 MB
  resnet50_static_moving_avg.pt                 22.99 MB
  resnet50_static_percentile.pt                 22.99 MB
  test.ipynb                                    0.01 MB
